# PatchTST — Walmart Store Sales Forecasting

**Architecture:** PatchTST — a Transformer for time series. It cuts each series into
fixed-length *patches* (like tokens), applies self-attention across patches, and uses
RevIN normalization so series of different scales are handled. Univariate in this
library (past sales only), but the required transformer for the project.

**Data view:** long format (`unique_id`, `ds`, `y`), from `src/features/nn_preprocessing.py`.
**Evaluation:** WMAE on the validation period (holiday weeks weighted 5x).
**MLflow structure:** experiment `PatchTST_Training` with runs `PatchTST_Data_Prep`,
`PatchTST_Baseline`, `PatchTST_Tuning`, `PatchTST_Best_Pipeline`.

**Result:** baseline (input_size=52) = 3,438 WMAE, tuning (input_size=104, finer
patches, bigger model) = 2,430 WMAE. The tuned config won and is kept as the best.

In [ ]:
# Run from the project root so data/ and src/ resolve, whether launched
# from the repo root or from the notebooks/ folder.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE

from src.features.nn_preprocessing import build_long_df, attach_static, temporal_split, FREQ
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Experiment tracking: DagsHub-hosted MLflow.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("PatchTST_Training")

SPLIT_DATE = "2012-01-01"
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape, "| stores:", stores.shape, "| features:", features.shape)

## Run 1 — Data preparation

In [ ]:
with mlflow.start_run(run_name="PatchTST_Data_Prep"):
    # Step 1: build the long panel and split by date.
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)

    # Step 2: log the preprocessing choices.
    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("gap_fill", "reindex_weekly_y0")
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_metric("train_rows", float(len(train_df)))
    mlflow.log_metric("valid_rows", float(len(valid_df)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon (weeks):", horizon)

## WMAE evaluation helper

In [ ]:
def evaluate_forecast(forecast_df, valid_df, model_col="PatchTST"):
    # Join forecasts to the true validation values on series id and date.
    merged = forecast_df.merge(
        valid_df[["unique_id", "ds", "y", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    # Neural models can output small negatives; sales can't be negative, so clip.
    preds = merged[model_col].clip(lower=0)
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2 — Baseline PatchTST

`input_size=52`, patches of length 16. A light transformer config (2 layers, 4 heads,
hidden 64) so it trains in reasonable time on CPU. **Result: WMAE 3,438.**

In [ ]:
with mlflow.start_run(run_name="PatchTST_Baseline"):
    params = dict(h=horizon, input_size=52, patch_len=16, stride=8,
                  encoder_layers=2, n_heads=4, hidden_size=64, linear_hidden_size=128,
                  max_steps=500, start_padding_enabled=True,
                  random_seed=SEED, alias="PatchTST")
    mlflow.log_params(params)

    nf = NeuralForecast(models=[PatchTST(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "PatchTST")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"PatchTST baseline WMAE: {wmae:,.2f}")

## Run 3 — Tuning

Longer lookback (`input_size=104`), finer patches, and a slightly larger model.
**Result: WMAE 2,430 — better than the baseline, so this config is kept.**

In [ ]:
with mlflow.start_run(run_name="PatchTST_Tuning"):
    params = dict(h=horizon, input_size=104, patch_len=8, stride=4,
                  encoder_layers=3, n_heads=8, hidden_size=128, linear_hidden_size=256,
                  max_steps=800, start_padding_enabled=True,
                  random_seed=SEED, alias="PatchTST")
    mlflow.log_params(params)

    nf = NeuralForecast(models=[PatchTST(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "PatchTST")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"PatchTST tuned WMAE: {wmae:,.2f}")

## Run 4 — Best pipeline (retrain on all data, save for inference)

The tuned config won (2,430 vs 3,438), so `BEST` is the tuned config. Retrain on the
full history and save for the inference step.

In [9]:
# Best config = the tuned run (lower WMAE).
BEST = dict(h=horizon, input_size=104, patch_len=8, stride=4,
            encoder_layers=3, n_heads=8, hidden_size=128, linear_hidden_size=256,
            max_steps=800, start_padding_enabled=True,
            random_seed=SEED, alias="PatchTST")

with mlflow.start_run(run_name="PatchTST_Best_Pipeline"):
    mlflow.log_params(BEST)

    # Retrain on the full history so the saved model uses every available week.
    nf_best = NeuralForecast(models=[PatchTST(loss=MAE(), **BEST)], freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])

    save_path = "saved_models/patchtst_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="patchtst_nf")
    print("Saved PatchTST model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 545 K  | train
-----------------------------------------------------------
545 K     Trainable params
3         Non-trainable params
545 K     Total params
2.180     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Epoch 0:  95%|█████████▌| 100/105 [03:13<00:09,  0.52it/s, v_num=22, train_loss_step=2.64e+3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  90%|█████████ | 95/105 [03:28<00:21,  0.46it/s, v_num=22, train_loss_step=1.95e+3, train_loss_epoch=3.3e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  86%|████████▌ | 90/105 [03:26<00:34,  0.44it/s, v_num=22, train_loss_step=2.8e+3, train_loss_epoch=2.69e+3]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  81%|████████  | 85/105 [02:54<00:41,  0.49it/s, v_num=22, train_loss_step=1.42e+3, train_loss_epoch=2.52e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  76%|███████▌  | 80/105 [02:43<00:50,  0.49it/s, v_num=22, train_loss_step=1.58e+3, train_loss_epoch=2.39e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |

`Trainer.fit` stopped: `max_steps=800` reached.


Epoch 7: 100%|██████████| 65/65 [02:13<00:00,  0.49it/s, v_num=22, train_loss_step=1.76e+3, train_loss_epoch=2.25e+3]
Saved PatchTST model to saved_models/patchtst_nf
🏃 View run PatchTST_Best_Pipeline at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/de189c10302d40c29d6e8a285fd374af
🧪 View experiment at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/2


## Notes

- PatchTST is univariate in this library (EXOGENOUS flags are False), so it also
  forecasts from past sales only. The 'do features help?' test is done separately with
  TFT, which does support exogenous variables.
- Tuned PatchTST (2,430) beats DLinear (2,679); N-BEATS generic (2,193) is still best.